# Kannon Ragendra Pipelining  : Data Ingestion to Vector

In [6]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


In [12]:
## Read all the pdfs inside the directory
def process_all_pdf(pdf_directory:str):
    """Process all documents in the given directory and return a list of langchain documents."""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}.")
    
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file.name}...")
        try:
            # Load the PDF document
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"
                
            all_documents.extend(documents)
            
            print(f"Finished processing {pdf_file.name}.")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    return all_documents

all_pdf_documents=process_all_pdf("../data/pdfs/")

Found 2 PDF files in ../data/pdfs/.
Processing nep212592.pdf...
Finished processing nep212592.pdf.
Processing Constitution-of-Nepal.pdf...
Finished processing Constitution-of-Nepal.pdf.


In [13]:
all_pdf_documents

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20221006092810', 'source': '../data/pdfs/nep212592.pdf', 'total_pages': 409, 'page': 0, 'page_label': '1', 'source_file': 'nep212592.pdf', 'file_type': 'pdf'}, page_content='http://www.lawcommission.gov.np \n1 \nThe Muluki Ain (General Code) \nDate of Authentication and Publication \n2019.12.30(…………..) \nAmending Acts: \n1. The Muluki (First Amendment) Act, 2021(…) 2021.6.17(……….) \n2. The Contract Act, 2023(…) 2023.6.9(……….) \n3. The Muluki (Second Amendment) Act, 2024(…) 2024.6.11(……….) \n4. The Muluki (Third Amendment) Act, 2025(…) 2025.7.9(……….) \n5. The Muluki (Fourth Amendment) Act, 2027(…) 2027.5.10(……….) \n6. The Muluki (Fifth Amendment) Act, 2031(…) 2031.6.20(……….) \n7. The Evidence Act, 2031(…) 2031.7.5(……….) \n8. The Muluki (Sixth Amendment) Act, 2033(…) 2033.4.28(……….) \n9. The Muluki (Seventh Amendment) Act, 2034(…) 2034.9.27(……….) \n10. The Muluki (Eighth Amendment) Act, 2042(…)  2042.6.23.4

In [21]:
## Text Splitting into chunks
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs= text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")
    if split_docs:
       print(f"\n Example Chunk:")
       print(f"Content: {split_docs[0].page_content[:500]}...")  # Print first 200 characters of the first chunk
       print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [23]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)

Split 649 documents into 1588 chunks.

 Example Chunk:
Content: http://www.lawcommission.gov.np 
1 
The Muluki Ain (General Code) 
Date of Authentication and Publication 
2019.12.30(…………..) 
Amending Acts: 
1. The Muluki (First Amendment) Act, 2021(…) 2021.6.17(……….) 
2. The Contract Act, 2023(…) 2023.6.9(……….) 
3. The Muluki (Second Amendment) Act, 2024(…) 2024.6.11(……….) 
4. The Muluki (Third Amendment) Act, 2025(…) 2025.7.9(……….) 
5. The Muluki (Fourth Amendment) Act, 2027(…) 2027.5.10(……….) 
6. The Muluki (Fifth Amendment) Act, 2031(…) 2031.6.20(……….) 
7...
Metadata: {'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20221006092810', 'source': '../data/pdfs/nep212592.pdf', 'total_pages': 409, 'page': 0, 'page_label': '1', 'source_file': 'nep212592.pdf', 'file_type': 'pdf'}


In [24]:
chunks

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20221006092810', 'source': '../data/pdfs/nep212592.pdf', 'total_pages': 409, 'page': 0, 'page_label': '1', 'source_file': 'nep212592.pdf', 'file_type': 'pdf'}, page_content='http://www.lawcommission.gov.np \n1 \nThe Muluki Ain (General Code) \nDate of Authentication and Publication \n2019.12.30(…………..) \nAmending Acts: \n1. The Muluki (First Amendment) Act, 2021(…) 2021.6.17(……….) \n2. The Contract Act, 2023(…) 2023.6.9(……….) \n3. The Muluki (Second Amendment) Act, 2024(…) 2024.6.11(……….) \n4. The Muluki (Third Amendment) Act, 2025(…) 2025.7.9(……….) \n5. The Muluki (Fourth Amendment) Act, 2027(…) 2027.5.10(……….) \n6. The Muluki (Fifth Amendment) Act, 2031(…) 2031.6.20(……….) \n7. The Evidence Act, 2031(…) 2031.7.5(……….) \n8. The Muluki (Sixth Amendment) Act, 2033(…) 2033.4.28(……….) \n9. The Muluki (Seventh Amendment) Act, 2034(…) 2034.9.27(……….) \n10. The Muluki (Eighth Amendment) Act, 2042(…)  2042.6.23.4

In [26]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

/home/sandip/Work/Personal/kanoon-ragendra/.venv/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 452.39it/s]


Model loaded successfully. Embedding dimension: 384


In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [30]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector database
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 1588 texts...


Batches: 100%|██████████| 50/50 [01:03<00:00,  1.28s/it]


Generated embeddings with shape: (1588, 384)
Adding 1588 documents to vector store...
Successfully added 1588 documents to vector store
Total documents in collection: 1588


In [31]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [43]:
rag_retriever.retrieve("What is Election Commission?")

Retrieving documents for query: 'What is Election Commission?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.08it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_08fb1213_1442',
  'content': '162 \n \n246. Functions, duties and powers of the Election  Commission: (1)  The Election \nCommission shal l, subject to this Constitution and the Federal l aw, conduct, \nsupervise, direct and control the election to the President, Vice -President, \nmembers of the Federal Parliament, members of State A ssemblies and \nmembers of Local l evel. For these purposes, the Election Commission shall \nprepare electoral rolls. \n(2) The Election Commission shall hold a referendum on a matter of \nnational importance pursuant to this Constitution and the Federal law \n(3) If, after nomination of candidacy for the President, Vic e-\nPresident, member of the Federal  Parliament, member of State  Assemblies or \nmember of Local l evel h as been filed but before the election  results are \ndeclared, a question arises about the qualification of a candidate, the Election \nCommission shall decide that question. \n(4) The Election Commission may so delegate

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.schema import HumanMessage, SystemMessage

In [58]:
# Check if the GROQ_API_KEY is set in the environment and which model is available for use

import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'model_card': 'openai/gpt-oss-safeguard-20b', 'id': 'openai/gpt-oss-safeguard-20b', 'object': 'model', 'created': 1761708789, 'owned_by': 'OpenAI', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 65536, 'hugging_face_id': 'openai/gpt-oss-safeguard-20b', 'name': 'Safety GPT OSS 20B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 65536, 'pricing': {'prompt': '0.000000075', 'completion': '0.0000003', 'image': '0', 'request': '0', 'input_cache_read': '0.0000000375'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['tools', 'json_mode', 'structured_outputs', 'reasoning']}, {'id': 'canopylabs/orpheus-arabic-saudi', 'object': 'model', 'created': 1765926439, 'owned_by': 'Canopy Labs', 'active': True, 'context_window': 4000, 'public_apps': None, 'max_completion_tokens': 50000, 'name': 'Canopy Labs Orphe

In [ ]:
class GroqLLM:
    def __init__(self, model_name: str = "openai/gpt-oss-20b", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (openai/gpt-oss-20b)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"


In [55]:
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: openai/gpt-oss-20b
Groq LLM initialized successfully!


In [56]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="openai/gpt-oss-20b",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [63]:
answer=rag_simple("Who is the executive head of the Nepal?",rag_retriever,llm)

Retrieving documents for query: 'Who is the executive head of the Nepal?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.19it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


In [64]:
answer

'The executive head of Nepal is the **Prime Minister**, who leads the Council of Ministers and exercises the executive power of the state.'